# Edge-AGI & Recursive Compression
## Evaluating Recursive Architectures Under Extreme Model Compression for Resource-Constrained Reasoning

**Research Question:** Can TRM maintain abstract reasoning capability when subjected to INT8/INT4 quantization and structured pruning, and does recursive depth provide an energy-efficient path to reasoning that survives compression?

---

### Notebook Structure
| Section | Description |
|---|---|
| 0 | Environment Setup & Repo Clone |
| 1 | Sudoku Dataset (standalone generator) |
| 2 | Minimal TRM Implementation (paper-faithful, no Hydra) |
| 3 | Full TRM Training via Repo (reference commands) |
| 4 | Quantization Wrappers (FP32 / INT8 / INT4) |
| 5 | **Main Experiment: Reasoning Decay Analysis** |
| 6 | Recursive Depth × Quantization Grid |
| 7 | Similarity-Score Loss (Novel Contribution) |
| 8 | Model Size & SRAM Footprint Estimator |

> **Runtime note:** Sections 0–3 perform real training. Set `SMOKE_TEST = True` (default) to run a short 200-epoch proof-of-concept on a tiny subset. Set `SMOKE_TEST = False` and point `CHECKPOINT_PATH` at a real trained checkpoint for the full quantization experiments.


In [19]:
import torch
# At the top of your notebook, before building the model:
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True



---
## Section 0 — Environment Setup

In [20]:
!git clone https://github.com/Seqaeon/EdgeTRM.git

Cloning into 'EdgeTRM'...
remote: Enumerating objects: 60, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 60 (delta 17), reused 57 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (60/60), 1.57 MiB | 15.56 MiB/s, done.
Resolving deltas: 100% (17/17), done.


In [21]:
!git pull origin main

From https://github.com/Seqaeon/EdgeTRM
 * branch            main       -> FETCH_HEAD
Already up to date.


In [22]:
import os
os.chdir("/root/EdgeTRM/TinyRecursiveModels")
print(os.getcwd())


/root/EdgeTRM/TinyRecursiveModels


In [23]:
# %uv pip install --use-pep517

In [24]:
# ── 0.1  Clone TRM repo (skip if already present) ──────────────────────────
import os, subprocess

REPO_DIR = "."#TinyRecursiveModels"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "https://github.com/SamsungSAILMontreal/TinyRecursiveModels.git"],
                   check=True)
print("Repo ready.")

# ── 0.2  Install dependencies ───────────────────────────────────────────────
# Torch is assumed present (Colab / local GPU).  Additional deps:
%uv pip install -q py-sudoku bitsandbytes matplotlib seaborn pandas

# Install TRM-specific requirements (minus torch lines)
import re, pathlib
req_text = pathlib.Path(f"{REPO_DIR}/requirements.txt").read_text()
# Filter out torch lines (already installed)
filtered = [l for l in req_text.splitlines()
            if l.strip() and not l.startswith("#") and "torch" not in l.lower()]
with open("/tmp/trm_reqs.txt", "w") as f:
    f.write("\n".join(filtered))
%uv pip install -q -r /tmp/trm_reqs.txt || true   # best-effort; some extras may not resolve

Repo ready.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [25]:
# ── 0.3  Global imports & config ────────────────────────────────────────────
import sys, time, copy, json, math, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import pandas as pd
warnings.filterwarnings("ignore")

# ── Global experiment flags ─────────────────────────────────────────────────
SMOKE_TEST       = False          # True → quick run for CI / sanity; False → full eval
DEVICE           = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED             = 42
CHECKPOINT_PATH  = None          # e.g. "checkpoints/trm_sudoku.pt"  ← set after full training

torch.manual_seed(SEED)
np.random.seed(SEED)
# print(f"Device: {DEVICE} | Smoke test: {SMOKE_TEST}")


# Add this at the top of the 'Global experiment flags' section
NUM_GPUS = torch.cuda.device_count()

# Add this helper function to the cell
def get_model(m):
    """Safely extracts the model from a DataParallel wrapper if present."""
    return m.module if isinstance(m, torch.nn.DataParallel) else m

# Update the final print statement
print(f"Device: {DEVICE} ({NUM_GPUS} GPUs available) | Smoke test: {SMOKE_TEST}")


Device: cuda (1 GPUs available) | Smoke test: False


---
## Section 1 — Sudoku Dataset

In [26]:
# ── 1.1  Sudoku puzzle generator ────────────────────────────────────────────
# We use py-sudoku to generate valid puzzles.
# Each puzzle is a (81,) tensor of tokens in [0..9]:
#   0 = empty cell, 1–9 = digit.
# Target is the fully solved grid.

from sudoku import Sudoku   # py-sudoku
from concurrent.futures import ProcessPoolExecutor
from functools import partial
import tqdm
def generate_sudoku_pair(difficulty: float = 0.6, seed: int = None):
    """Return (puzzle_flat, solution_flat) as numpy int arrays of shape (81,)."""
    rng = np.random.default_rng(seed)
    # py-sudoku's internal seed is not exposed cleanly; generate many and pick
    puzzle = Sudoku(3, seed=int(rng.integers(0, 1_000_000))).difficulty(difficulty)
    solution = puzzle.solve()
    if solution.board is None:
        return None
    def board_to_flat(board):
        return np.array([0 if v is None else v
                         for row in board for v in row], dtype=np.int64)
    return board_to_flat(puzzle.board), board_to_flat(solution.board)


# def make_dataset(n_puzzles: int, difficulty: float = 0.6, verbose: bool = True):
#     puzzles, solutions = [], []
#     attempts = 0
#     while len(puzzles) < n_puzzles:
#         result = generate_sudoku_pair(difficulty, seed=attempts)
#         attempts += 1
#         if result is None:
#             continue
#         p, s = result
#         puzzles.append(p)
#         solutions.append(s)
#     puzzles   = np.stack(puzzles)    # (N, 81)
#     solutions = np.stack(solutions)  # (N, 81)
#     if verbose:
#         print(f"Generated {n_puzzles} puzzles (attempts={attempts})")
#     return puzzles, solutions






def make_dataset(n_puzzles, difficulty=0.6, n_workers=8,  verbose: bool = True):
    worker = partial(generate_sudoku_pair, difficulty)   # ← replaces the lambda
    with ProcessPoolExecutor(max_workers=n_workers) as ex:
        results = list(tqdm.tqdm(
            ex.map(worker, range(n_puzzles * 2)),
            total=n_puzzles * 2
        ))
    valid = [r for r in results if r is not None][:n_puzzles]
    puzzles   = np.stack([v[0] for v in valid])
    solutions = np.stack([v[1] for v in valid])
    return puzzles, solutions
class SudokuDataset(Dataset):
    def __init__(self, puzzles: np.ndarray, solutions: np.ndarray,
                 n_aug: int = 1):
        """
        n_aug: simple digit-permutation augmentation count.
        The TRM paper uses 1000 augmentations; we keep it low for smoke tests.
        """
        self.puzzles   = torch.tensor(puzzles,   dtype=torch.long)
        self.solutions = torch.tensor(solutions, dtype=torch.long)
        self.n_aug     = n_aug

    def _permute_digits(self, puzzle, solution, seed):
        """Randomly permute digit labels 1–9 (valid Sudoku symmetry)."""
        rng   = np.random.default_rng(seed)
        perm  = np.array([0] + list(rng.permutation(9) + 1), dtype=np.int64)
        return perm[puzzle], perm[solution]

    def __len__(self):
        return len(self.puzzles) * self.n_aug

    def __getitem__(self, idx):
        base   = idx % len(self.puzzles)
        aug_id = idx // len(self.puzzles)
        p      = self.puzzles[base].numpy()
        s      = self.solutions[base].numpy()
        if aug_id > 0:
            p, s = self._permute_digits(p, s, seed=aug_id * 997 + base)
        return torch.tensor(p), torch.tensor(s)


# ── 1.2  Generate train / val / test splits ──────────────────────────────────
N_TRAIN = 200 if SMOKE_TEST else 1000
N_VAL   =  50 if SMOKE_TEST else  200
N_TEST  = 200 if SMOKE_TEST else 1000
N_AUG   =   1 if SMOKE_TEST else  10 #50
log_batch_interval = 1# max(1, N_EPOCHS // 10)  # = max(1, 5000//10) = 500
print("Generating train set...")
train_p, train_s = make_dataset(N_TRAIN)
print("Generating val set...")
val_p,   val_s   = make_dataset(N_VAL,  verbose=False)
print("Generating test set...")
test_p,  test_s  = make_dataset(N_TEST, verbose=False)

train_ds  = SudokuDataset(train_p, train_s, n_aug=N_AUG)
val_ds    = SudokuDataset(val_p,   val_s,   n_aug=1)
test_ds   = SudokuDataset(test_p,  test_s,  n_aug=1)

BATCH = 128
train_loader = DataLoader(
    train_ds, batch_size=BATCH, shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True   # keep workers alive between epochs
)
# train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=0)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

Generating train set...


100%|█████████████████████████████████████████████████████████████████████| 2000/2000 [00:03<00:00, 655.73it/s]

Generating val set...



100%|███████████████████████████████████████████████████████████████████████| 400/400 [00:01<00:00, 315.06it/s]

Generating test set...



100%|█████████████████████████████████████████████████████████████████████| 2000/2000 [00:06<00:00, 321.28it/s]

Train: 10000 | Val: 200 | Test: 1000


---
## Section 2 — Minimal TRM Implementation

This is a standalone, paper-faithful re-implementation of **TRM-MLP** (the best Sudoku variant).
It does **not** use Hydra / the repo's training loop, making it fully self-contained
for the quantization experiments in Sections 4–8.

Key design choices (from paper §4):
- Single 2-layer network (MLP-Mixer style, no self-attention for fixed L=81)
- Carry `(y, z)` across deep supervision steps
- `n=6` inner recursions, `T=3` outer recursion passes (T-1 no_grad + 1 with_grad)
- EMA of weights (decay=0.999)
- Simplified ACT: binary CE on halt probability, no extra forward pass

In [27]:
# ── 2.1  Model components ────────────────────────────────────────────────────

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps    = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).rsqrt()
        return x * rms * self.weight


class SwiGLU(nn.Module):
    """Token-mixing MLP with SwiGLU (replaces self-attention for small fixed L)."""
    def __init__(self, seq_len: int, expansion: int = 4):
        super().__init__()
        hidden = seq_len * expansion
        self.gate  = nn.Linear(seq_len, hidden, bias=False)
        self.value = nn.Linear(seq_len, hidden, bias=False)
        self.proj  = nn.Linear(hidden, seq_len, bias=False)

    def forward(self, x):           # x: (B, L, D)
        x = x.transpose(1, 2)       # (B, D, L)
        x = self.proj(F.silu(self.gate(x)) * self.value(x))
        return x.transpose(1, 2)    # (B, L, D)


class ChannelMLP(nn.Module):
    """Standard channel-mixing MLP (operates on D dimension)."""
    def __init__(self, dim: int, expansion: int = 4):
        super().__init__()
        hidden = dim * expansion
        self.gate  = nn.Linear(dim, hidden, bias=False)
        self.value = nn.Linear(dim, hidden, bias=False)
        self.proj  = nn.Linear(hidden, dim,  bias=False)

    def forward(self, x):
        return self.proj(F.silu(self.gate(x)) * self.value(x))


class TRMBlock(nn.Module):
    """Single TRM transformer-like block (token-mix → channel-mix with residuals)."""
    def __init__(self, seq_len: int, dim: int):
        super().__init__()
        self.norm1   = RMSNorm(dim)
        self.tok_mix = SwiGLU(seq_len)
        self.norm2   = RMSNorm(dim)
        self.ch_mix  = ChannelMLP(dim)

    def forward(self, x):
        x = x + self.tok_mix(self.norm1(x))
        x = x + self.ch_mix(self.norm2(x))
        return x


class TinyRecursiveModel(nn.Module):
    """
    Paper-faithful TRM-MLP for fixed-length tasks (Sudoku 9×9 = L=81 tokens).

    Forward logic (mirrors Algorithm 3 / Figure 3 of the paper):
      latent_recursion(x, y, z):
          for i in range(n):  z = net(concat(x, y, z))
          y = net(concat(y, z))
          return y, z

      deep_recursion (T-1 no_grad + 1 with_grad) → y_hat, q_hat

    Training loop uses N_sup deep supervision steps with carry (y, z).
    """
    def __init__(self,
                 vocab_size:  int = 10,   # 0=empty, 1–9
                 seq_len:     int = 81,
                 hidden_size: int = 128,  # 512 in the paper; 128 for smoke tests
                 n_layers:    int = 2,
                 n_cycles:    int = 6,    # inner recursions (n)
                 T_cycles:    int = 3):   # outer passes (T)
        super().__init__()
        self.vocab_size  = vocab_size
        self.seq_len     = seq_len
        self.hidden_size = hidden_size
        self.n_cycles    = n_cycles
        self.T_cycles    = T_cycles

        # Embeddings
        self.x_embed = nn.Embedding(vocab_size, hidden_size)  # input
        self.y_embed = nn.Embedding(vocab_size, hidden_size)  # answer
        self.z_init  = nn.Parameter(torch.zeros(1, seq_len, hidden_size))  # learned z₀

        # Input projection: concatenation of x, y, z → hidden
        self.in_proj = nn.Linear(hidden_size * 3, hidden_size, bias=False)

        # Shared recursive network  fL  (n_layers blocks)
        self.f_latent = nn.Sequential(*[TRMBlock(seq_len, hidden_size)
                                        for _ in range(n_layers)])

        # Answer-update projection: concat(y, z) → hidden  (fH equivalent)
        self.y_proj   = nn.Linear(hidden_size * 2, hidden_size, bias=False)
        self.f_answer = nn.Sequential(*[TRMBlock(seq_len, hidden_size)
                                        for _ in range(n_layers)])

        # Output head
        self.lm_head  = nn.Linear(hidden_size, vocab_size, bias=False)
        # ACT halt head
        self.halt_head = nn.Linear(hidden_size, 1, bias=False)

    # ── Core recursion steps ─────────────────────────────────────────────────
    def _latent_step(self, x_emb, y_emb, z):
        """One inner recursion: z ← fL(x, y, z)."""
        inp = self.in_proj(torch.cat([x_emb, y_emb, z], dim=-1))
        return self.f_latent(inp)

    def _answer_step(self, y_emb, z):
        """Answer update: y ← fH(y, z)."""
        inp = self.y_proj(torch.cat([y_emb, z], dim=-1))
        return self.f_answer(inp)

    def latent_recursion(self, x_emb, y_emb, z):
        """Run n_cycles latent steps then one answer update. Returns new y_emb, z."""
        for _ in range(self.n_cycles):
            z = self._latent_step(x_emb, y_emb, z)
        y_emb = self._answer_step(y_emb, z)
        return y_emb, z

    def decode(self, y_emb):
        """Map y embedding → logits (B, L, V) and hard token predictions."""
        logits   = self.lm_head(y_emb)           # (B, L, V)
        y_tokens = logits.argmax(-1)              # (B, L)
        return logits, y_tokens

    # def halt_prob(self, y_emb):
    #     """Scalar halt probability per batch item."""
    #     return torch.sigmoid(self.halt_head(y_emb.mean(1))).squeeze(-1)  # (B,)
    # In TinyRecursiveModel.halt_prob — return raw logits instead of sigmoid output:
    def halt_prob(self, y_emb):
        return self.halt_head(y_emb.mean(1)).squeeze(-1)   # no sigmoid
    # ── Deep recursion (T-1 no_grad + 1 with_grad) ──────────────────────────
    def deep_recursion(self, x_emb, y_emb, z):
        with torch.no_grad():
            for _ in range(self.T_cycles - 1):
                y_emb, z = self.latent_recursion(x_emb, y_emb, z)
        y_emb, z = self.latent_recursion(x_emb, y_emb, z)  # with gradients
        return y_emb, z

    # ── Main forward ─────────────────────────────────────────────────────────
    def forward(self, x_tokens, y_tokens_in, z_carry):
        """
        x_tokens:   (B, L)  input puzzle
        y_tokens_in:(B, L)  previous answer tokens (or zeros for first step)
        z_carry:    (B, L, D) carried latent from previous supervision step
        Returns: logits, y_tokens_out, new_z_carry, halt_prob
        """
        x_emb = self.x_embed(x_tokens)       # (B, L, D)
        y_emb = self.y_embed(y_tokens_in)     # (B, L, D)

        y_emb, z = self.deep_recursion(x_emb, y_emb, z_carry)

        logits, y_tokens_out = self.decode(y_emb)
        q = self.halt_prob(y_emb)
        return logits, y_tokens_out, z.detach(), q

    def init_carry(self, batch_size):
        return self.z_init.expand(batch_size, -1, -1).clone()


# ── Quick parameter count check ──────────────────────────────────────────────
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

H   = 64 if SMOKE_TEST else 512
# trm = TinyRecursiveModel(hidden_size=H).to(DEVICE)
# n_p = count_params(trm)



# Replace the existing trm instantiation with this:
trm = TinyRecursiveModel(hidden_size=H)
if NUM_GPUS > 1:
    trm = torch.nn.DataParallel(trm)
trm = trm.to(DEVICE)

# Use the helper for parameter counting
n_p = count_params(get_model(trm))
print(f"TRM parameters: {n_p:,}  ({n_p/1e6:.2f}M)")
print(f"FP32 size: {n_p * 4 / 1024:.1f} KB")

TRM parameters: 14,270,000  (14.27M)
FP32 size: 55742.2 KB


In [28]:
# ── 2.2  EMA helper ──────────────────────────────────────────────────────────

# class EMA:
#     """Exponential Moving Average of model weights (paper uses 0.999)."""
#     def __init__(self, model: nn.Module, decay: float = 0.999):
#         self.model  = model
#         self.decay  = decay
#         self.shadow = copy.deepcopy(model.state_dict())

#     @torch.no_grad()
#     def update(self):
#         for k, v in self.model.state_dict().items():
#             if v.dtype.is_floating_point:
#                 self.shadow[k] = self.decay * self.shadow[k] + (1 - self.decay) * v

#     def apply_shadow(self):
#         self._backup = copy.deepcopy(self.model.state_dict())
#         self.model.load_state_dict(self.shadow)

#     def restore(self):
#         self.model.load_state_dict(self._backup)
class EMA:
    def __init__(self, model: nn.Module, decay: float = 0.999):
        self.model  = model
        self.decay  = decay
        # Store shadow params as a list, not a state_dict copy
        self.shadow = [p.data.clone() for p in model.parameters()]

    @torch.no_grad()
    def update(self):
        # Single vectorized kernel across ALL params — much faster than a Python loop
        torch._foreach_lerp_(
            self.shadow,
            [p.data for p in self.model.parameters()],
            1.0 - self.decay
        )

    def apply_shadow(self):
        self._backup = [p.data.clone() for p in self.model.parameters()]
        for p, s in zip(self.model.parameters(), self.shadow):
            p.data.copy_(s)

    def restore(self):
        for p, b in zip(self.model.parameters(), self._backup):
            p.data.copy_(b)

In [29]:
# ── 2.3  Training loop ───────────────────────────────────────────────────────

def train_trm(model: TinyRecursiveModel,
              train_loader: DataLoader,
              val_loader:   DataLoader,
              n_epochs:     int   = 200,
              lr:           float = 1e-4 * (BATCH / 64),
              weight_decay: float = 1.0,
              n_sup:        int   = 8,    # max deep supervision steps (16 in paper)
              log_interval: int   = 20,
              ema_decay:    float = 0.999,
              save_path:    str   = "trm_ckpt.pt"):
    """
    Deep Supervision training loop.
    n_sup: max supervision steps per sample (paper uses 16; 8 here for speed).
    """
    # opt = torch.optim.AdamW(model.parameters(), lr=lr,
    #                         weight_decay=weight_decay, betas=(0.9, 0.95))
    opt = torch.optim.AdamW(
        trm.parameters(),
        lr=lr,   # linear LR scaling with batch
        weight_decay=weight_decay,
        betas=(0.9, 0.95),
        fused=True                 # single CUDA kernel for all param updates
    )
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)
    ema   = EMA(model, decay=ema_decay)
    scaler = None  # no GradScaler needed for BF16

    history = {"train_loss": [], "val_acc": [], "val_cell_acc": []}
    best_val_acc = 0.0

    for epoch in range(1, n_epochs + 1):
        model.train()
        epoch_loss = 0.0

        for batch_idx, (x_batch, y_true) in enumerate(train_loader):
            x_batch = x_batch.to(DEVICE)    # (B, 81)
            y_true  = y_true.to(DEVICE)     # (B, 81)
            B = x_batch.size(0)

            # Update these lines inside the loop
            y_carry = torch.zeros(B, get_model(model).seq_len, dtype=torch.long, device=DEVICE)
            z_carry = get_model(model).init_carry(B).to(DEVICE)




            total_loss = torch.tensor(0.0, device=DEVICE)
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
    
                for step in range(n_sup):
                    logits, y_pred, z_carry, q = model(x_batch, y_carry, z_carry)
                    # ── Classification loss ──────────────────────────────────────
                    # Update the cross_entropy line
                    ce_loss = F.cross_entropy(logits.reshape(-1, get_model(model).vocab_size), y_true.reshape(-1))
                    # ── Simplified ACT loss (§4.6) ───────────────────────────────
                    correct = (y_pred == y_true).all(dim=-1).float()  # (B,)
                    # act_loss = F.binary_cross_entropy(q, correct)
                    # Loss — replace binary_cross_entropy with binary_cross_entropy_with_logits:
                    act_loss = F.binary_cross_entropy_with_logits(q, correct)
                    total_loss = total_loss + ce_loss + 0.1 * act_loss
    
                    # Early stopping if ACT signals halt
                    y_carry = y_pred.detach()
                    # Early-stopping check — sigmoid is now needed here since q is raw logits:
                    if torch.sigmoid(q).mean().item() > 0.5 and step > 0:
                        break
                    # if q.mean().item() > 0.5 and step > 0:
                    #     break

            opt.zero_grad()
            total_loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            ema.update()
            epoch_loss += total_loss.item()
            if batch_idx % log_batch_interval == 0:
                print(f"Epoch {epoch}/{n_epochs} | Batch {batch_idx}/{len(train_loader)} | "
                      f"Loss: {total_loss.item():.4f}", flush=True)

        sched.step()
        avg_loss = epoch_loss / len(train_loader)
        history["train_loss"].append(avg_loss)

        if epoch % log_interval == 0 or epoch == n_epochs:
            # ── Evaluate with EMA weights ────────────────────────────────────
            ema.apply_shadow()
            exact_acc, cell_acc = evaluate(model, val_loader)
            ema.restore()

            history["val_acc"].append(exact_acc)
            history["val_cell_acc"].append(cell_acc)
            print(f"Epoch {epoch:4d}/{n_epochs} | loss={avg_loss:.4f} "
                  f"| val_exact={exact_acc:.3f} | val_cell={cell_acc:.3f}")

            if exact_acc > best_val_acc:
                best_val_acc = exact_acc
                torch.save({
                    "model": model.state_dict(),
                    "ema":   ema.shadow,
                    "epoch": epoch,
                    "val_acc": exact_acc
                }, save_path)

    print(f"\nBest val exact accuracy: {best_val_acc:.3f}")
    return history, ema


@torch.no_grad()
def evaluate(model: TinyRecursiveModel,
             loader:  DataLoader,
             n_sup:   int = 16) -> tuple:
    """
    Returns (exact_match_accuracy, cell_level_accuracy).
    exact_match: entire 81-cell grid must be correct.
    cell_level:  fraction of individual cells correct.
    """
    model.eval()
    exact_correct = cell_correct = total_cells = total_puzzles = 0

    for x_batch, y_true in loader:
        x_batch = x_batch.to(DEVICE)
        y_true  = y_true.to(DEVICE)
        B = x_batch.size(0)
        # Update the carry initialization
        y_carry = torch.zeros(B, get_model(model).seq_len, dtype=torch.long, device=DEVICE)
        z_carry = get_model(model).init_carry(B).to(DEVICE)
        



        for step in range(n_sup):
            _, y_pred, z_carry, q = model(x_batch, y_carry, z_carry)
            y_carry = y_pred
            if torch.sigmoid(q).mean().item() > 0.5 and step > 0:
                break
            # if q.mean().item() > 0.5 and step > 0:
            #     break

        exact_correct  += (y_pred == y_true).all(dim=-1).sum().item()
        cell_correct   += (y_pred == y_true).sum().item()
        total_puzzles  += B
        # Update the total_cells line
        total_cells += B * get_model(model).seq_len
    return exact_correct / total_puzzles, cell_correct / total_cells

In [30]:
from models.recursive_reasoning.trm import TinyRecursiveReasoningModel_ACTV1


In [31]:
# # 1. Build the model with the same config
# # CHECKPOINT_PATH = "/kaggle/input/datasets/cpmpml/arc-prize-trm-031/step_275886"

# # from models.trm import TRMModel   # or whatever class is used
# model_cfg = {
#     # "L_layers": 2,
#     # "H_cycles": 4,
#     # "L_cycles": 4,
#     # "halt_max_steps": 10,
#     # "batch_size": 128,
#     # "vocab_size": train_metadata.vocab_size,
#     # "seq_len": train_metadata.seq_len,
#     # "num_puzzle_identifiers": train_metadata.num_puzzle_identifiers,
#     # "causal": False
# }
# trm = TRMModel(model_cfg)

# # trm = TinyRecursiveReasoningModel_ACTV1(model_cfg)

# # 2. Load checkpoint
# ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
# trm.load_state_dict(ckpt)

# # 3. Use the model
# trm.eval()


In [ ]:
# ── 2.4  Run training ────────────────────────────────────────────────────────
N_EPOCHS = 200 if SMOKE_TEST else 5000
# CHECKPOINT_PATH = "/kaggle/input/datasets/cpmpml/arc-prize-trm-031/step_275886"
if CHECKPOINT_PATH and os.path.isfile(CHECKPOINT_PATH):
    print(f"Loading checkpoint from {CHECKPOINT_PATH}")
    ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    trm.load_state_dict(ckpt['ema'])   # use EMA weights directly
    history = None
else:
    print(f"Training TRM for {N_EPOCHS} epochs...")
    t0 = time.time()
    history, ema = train_trm(trm, train_loader, val_loader,
                              n_epochs=N_EPOCHS,
                              log_interval=1, #max(1, N_EPOCHS // 10),
                              save_path="trm_ckpt.pt")
    print(f"Training time: {(time.time()-t0)/60:.1f} min")
    # Load best checkpoint (EMA weights)
    ckpt = torch.load("trm_ckpt.pt", map_location=DEVICE)
    trm.load_state_dict(ckpt["ema"])

fp32_exact, fp32_cell = evaluate(trm, test_loader)
print(f"\nFP32 Test | exact: {fp32_exact:.4f} | cell: {fp32_cell:.4f}")

Training TRM for 5000 epochs...
Epoch 1/5000 | Batch 0/79 | Loss: 4.7763
Epoch 1/5000 | Batch 1/79 | Loss: 4.7619
Epoch 1/5000 | Batch 2/79 | Loss: 4.7450
Epoch 1/5000 | Batch 3/79 | Loss: 4.7326
Epoch 1/5000 | Batch 4/79 | Loss: 18.7838
Epoch 1/5000 | Batch 5/79 | Loss: 18.7506
Epoch 1/5000 | Batch 6/79 | Loss: 18.6755
Epoch 1/5000 | Batch 7/79 | Loss: 18.6345
Epoch 1/5000 | Batch 8/79 | Loss: 18.5663
Epoch 1/5000 | Batch 9/79 | Loss: 18.5211
Epoch 1/5000 | Batch 10/79 | Loss: 18.4780
Epoch 1/5000 | Batch 11/79 | Loss: 18.4188
Epoch 1/5000 | Batch 12/79 | Loss: 18.3439
Epoch 1/5000 | Batch 13/79 | Loss: 18.2882
Epoch 1/5000 | Batch 14/79 | Loss: 18.2472
Epoch 1/5000 | Batch 15/79 | Loss: 18.1922
Epoch 1/5000 | Batch 16/79 | Loss: 18.1269
Epoch 1/5000 | Batch 17/79 | Loss: 18.0841
Epoch 1/5000 | Batch 18/79 | Loss: 18.0137
Epoch 1/5000 | Batch 19/79 | Loss: 17.9682
Epoch 1/5000 | Batch 20/79 | Loss: 17.9215
Epoch 1/5000 | Batch 21/79 | Loss: 17.8623
Epoch 1/5000 | Batch 22/79 | Loss: 1

In [ ]:
# ── 2.5  Plot training curves ─────────────────────────────────────────────────
if history:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(history["train_loss"])
    ax1.set_title("Train Loss")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.set_yscale("log")
    ax1.grid(True, alpha=0.3)

    eval_epochs = [i * max(1, N_EPOCHS // 10) for i in range(1, len(history["val_acc"]) + 1)]
    ax2.plot(eval_epochs, history["val_acc"],      label="Exact Match",  marker="o")
    ax2.plot(eval_epochs, history["val_cell_acc"], label="Cell Accuracy", marker="s")
    ax2.set_title("Validation Accuracy")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Accuracy")
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
    plt.show()

---
## Section 3 — Full TRM Training via Repo (Reference)

The cells below are **reference commands** for training the full TRM using the official repo.
These require a proper GPU and produce the ~87% checkpoint used in the paper.
Skip this section if you already have a checkpoint.

In [ ]:
# ── 3.1  Official TRM-MLP Sudoku-Extreme training command ───────────────────
# Run from the TinyRecursiveModels directory.
# Expected: ~87% exact accuracy | Runtime: ~18h on 1x L40S (48 GB)

OFFICIAL_TRAIN_CMD = """
cd TinyRecursiveModels

# 1. Build the dataset (1000 puzzles, 1000 augmentations)
python dataset/build_sudoku_dataset.py \\
    --output-dir data/sudoku-extreme-1k-aug-1000 \\
    --subsample-size 1000 \\
    --num-aug 1000

# 2. Train
python pretrain.py \\
    arch=trm \\
    data_paths="[data/sudoku-extreme-1k-aug-1000]" \\
    evaluators="[]" \\
    epochs=50000 eval_interval=5000 \\
    lr=1e-4 puzzle_emb_lr=1e-4 \\
    weight_decay=1.0 puzzle_emb_weight_decay=1.0 \\
    arch.mlp_t=True arch.pos_encodings=none \\
    arch.L_layers=2 \\
    arch.H_cycles=3 arch.L_cycles=6 \\
    +run_name=trm_mlp_sudoku \\
    ema=True
"""
print(OFFICIAL_TRAIN_CMD)

# ── 3.2  Loading a repo checkpoint into this notebook ───────────────────────
# If you ran the above, set CHECKPOINT_PATH to the .pt file and re-run Section 2.
# The repo saves checkpoints under: TinyRecursiveModels/outputs/<run_name>/

# Example:
# CHECKPOINT_PATH = "TinyRecursiveModels/outputs/trm_mlp_sudoku/checkpoints/best.pt"
# Note: the repo uses its own model class; loading requires either:
#   (a) Using the repo's evaluate.py script directly, or
#   (b) Mapping the state_dict keys to our TinyRecursiveModel above.

---
## Section 4 — Quantization Wrappers

We implement three compression levels:
| Level | Method | PyTorch API | Expected size (7M params) |
|---|---|---|---|
| **FP32** | Baseline | — | ~27 MB |
| **INT8** | Dynamic post-training quantization | `torch.quantization.quantize_dynamic` | ~7 MB |
| **INT4** | Fake quantization (symmetric, per-tensor) | Custom wrapper | ~3.5 MB |

> **True INT4** storage requires custom CUDA kernels (e.g., `bitsandbytes`).  
> Here we use **fake quantization** to simulate INT4 arithmetic fidelity within FP32 storage — this measures reasoning quality, not memory, which is the key variable for reasoning decay.


In [ ]:
# ── 4.1  INT8 Dynamic Quantization ──────────────────────────────────────────

def quantize_int8(model: nn.Module) -> nn.Module:
    """Apply PyTorch dynamic INT8 quantization to all Linear layers."""
    q_model = copy.deepcopy(model).cpu()
    q_model = torch.quantization.quantize_dynamic(
        q_model,
        {nn.Linear},
        dtype=torch.qint8
    )
    return q_model


# ── 4.2  INT4 Fake Quantization ──────────────────────────────────────────────

class FakeQuantINT4(nn.Module):
    """Wraps a Linear layer and applies symmetric fake INT4 quantization."""
    def __init__(self, linear: nn.Linear):
        super().__init__()
        self.linear = linear

    def _fake_quant(self, x: torch.Tensor, n_bits: int = 4) -> torch.Tensor:
        """Round-to-nearest fake quantization, symmetric, per-tensor."""
        q_max = 2 ** (n_bits - 1) - 1          # 7 for INT4
        scale = x.abs().max().clamp(min=1e-8) / q_max
        x_q   = torch.clamp((x / scale).round(), -q_max, q_max)
        return x_q * scale                      # dequantize back to FP32

    def forward(self, x):
        w_q = self._fake_quant(self.linear.weight)
        return F.linear(x, w_q, self.linear.bias)


def quantize_int4_fake(model: nn.Module) -> nn.Module:
    """Replace all Linear layers with FakeQuantINT4 wrappers."""
    model_copy = copy.deepcopy(model)
    for name, module in list(model_copy.named_modules()):
        if isinstance(module, nn.Linear):
            # Navigate to parent and replace
            parts  = name.split(".")
            parent = model_copy
            for p in parts[:-1]:
                parent = getattr(parent, p)
            setattr(parent, parts[-1], FakeQuantINT4(module))
    return model_copy


# ── 4.3  Model size estimator ─────────────────────────────────────────────────

def estimate_model_size_kb(model: nn.Module, bits: int = 32) -> float:
    """Estimate parameter storage in KB at given precision."""
    n_params = sum(p.numel() for p in model.parameters())
    return n_params * bits / 8 / 1024


def get_actual_size_kb(model: nn.Module) -> float:
    """Get actual serialised size in KB by counting quantized buffers."""
    import io
    buf = io.BytesIO()
    torch.save(model.state_dict(), buf)
    return buf.tell() / 1024


# ── 4.4  Instantiate all model variants ──────────────────────────────────────
trm.cpu()   # move to CPU for quantization (PyTorch dynamic quant requires CPU)

trm_fp32 = trm
trm_int8 = quantize_int8(trm)
trm_int4 = quantize_int4_fake(trm)

# Summary
print("Model Variant | Theoretical Size | Actual .pt Size")
print("-" * 55)
for name, m, bits in [("FP32", trm_fp32, 32), ("INT8", trm_int8, 8), ("INT4 (fake)", trm_int4, 4)]:
    theo = estimate_model_size_kb(trm_fp32, bits)   # always relative to fp32 param count
    actual = get_actual_size_kb(m)
    print(f"  {name:<12} | {theo:>8.1f} KB        | {actual:>8.1f} KB")

print(f"\nTarget: <1 MB (1024 KB) SRAM")

---
## Section 5 — Main Experiment: Reasoning Decay Analysis

**Hypothesis:** There is a critical quantization threshold below which reasoning fails **catastrophically** (model forgets the game rules entirely) rather than **gracefully** (partial accuracy preserved).

We measure two metrics per quantization level:
- `exact_acc`: full puzzle must be solved — captures hard logical consistency
- `cell_acc`: fraction of individual cells correct — captures graceful degradation

In [ ]:
# ── 5.1  Evaluate all quantization levels ────────────────────────────────────

@torch.no_grad()
def evaluate_cpu(model: nn.Module, loader: DataLoader,
                 trm_ref: TinyRecursiveModel,
                 n_sup: int = 16) -> tuple:
    """
    Evaluation loop that works for both FP32 and quantized (CPU) models.
    We re-use the TRM logic but apply the quantized model for the forward pass.
    """
    model.eval()
    exact_correct = cell_correct = total_cells = total_puzzles = 0

    for x_batch, y_true in loader:
        # Stay on CPU for quantized models
        B = x_batch.size(0)
        # Update the carry lines inside evaluate_cpu
        y_carry = torch.zeros(B, get_model(trm_ref).seq_len, dtype=torch.long)
        z_carry = get_model(trm_ref).z_init.expand(B, -1, -1).clone().cpu()


        # For INT8 models we call model.forward() directly if it's a TRM
        # (dynamic quant only quantizes Linear weights, structure is preserved)
        try:
            for step in range(n_sup):
                logits, y_pred, z_carry, q = model(x_batch, y_carry, z_carry)
                y_carry = y_pred
                if q.mean().item() > 0.5 and step > 0:
                    break
        except Exception as e:
            print(f"  Warning: {e}")
            break

        exact_correct  += (y_pred == y_true).all(dim=-1).sum().item()
        cell_correct   += (y_pred == y_true).sum().item()
        total_puzzles  += B
        total_cells    += B * trm_ref.seq_len

    if total_puzzles == 0:
        return 0.0, 0.0
    return exact_correct / total_puzzles, cell_correct / total_cells


print("Evaluating quantization variants on test set...\n")
trm.cpu()
results = {}
for name, m in [("FP32", trm_fp32), ("INT8", trm_int8), ("INT4 (fake)", trm_int4)]:
    t0 = time.time()
    exact, cell = evaluate_cpu(m, test_loader, trm_ref=trm)
    elapsed = time.time() - t0
    results[name] = {"exact_acc": exact, "cell_acc": cell, "latency_s": elapsed / len(test_ds)}
    print(f"  {name:<14} | exact={exact:.4f} | cell={cell:.4f} | "
          f"latency={elapsed/len(test_ds)*1000:.2f} ms/puzzle")

# ── Reasoning Decay (delta from FP32) ────────────────────────────────────────
print("\n── Reasoning Decay (drop from FP32) ──────────────────────────────")
fp32_exact_test = results["FP32"]["exact_acc"]
fp32_cell_test  = results["FP32"]["cell_acc"]
for name, r in results.items():
    if name == "FP32":
        continue
    d_exact = fp32_exact_test - r["exact_acc"]
    d_cell  = fp32_cell_test  - r["cell_acc"]
    mode    = "CATASTROPHIC" if d_exact > 0.5 else ("GRACEFUL" if d_exact < 0.2 else "MODERATE")
    print(f"  {name:<14} | Δexact={d_exact:+.4f} | Δcell={d_cell:+.4f} | Mode: {mode}")

In [ ]:
# ── 5.2  Visualise reasoning decay ───────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

names       = list(results.keys())
exact_accs  = [results[n]["exact_acc"]  for n in names]
cell_accs   = [results[n]["cell_acc"]   for n in names]
latencies   = [results[n]["latency_s"] * 1000 for n in names]
bit_widths  = [32, 8, 4]
colors      = ["#2196F3", "#4CAF50", "#FF5722"]

# Plot 1: Exact match accuracy vs precision
axes[0].bar(names, exact_accs, color=colors, alpha=0.85, edgecolor="black", linewidth=0.8)
axes[0].set_title("Exact Match Accuracy", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Accuracy")
axes[0].set_ylim(0, 1)
axes[0].axhline(0.5, ls="--", c="red", alpha=0.5, label="50% threshold")
axes[0].legend()
for i, v in enumerate(exact_accs):
    axes[0].text(i, v + 0.02, f"{v:.3f}", ha="center", fontsize=11)

# Plot 2: Graceful vs catastrophic — exact vs cell
x   = np.arange(len(names))
w   = 0.35
axes[1].bar(x - w/2, exact_accs, w, label="Exact Match",  color=colors, alpha=0.8, edgecolor="black")
axes[1].bar(x + w/2, cell_accs,  w, label="Cell Accuracy", color=colors, alpha=0.4, edgecolor="black",
            hatch="//")
axes[1].set_title("Graceful vs Catastrophic Decay", fontsize=13, fontweight="bold")
axes[1].set_xticks(x)
axes[1].set_xticklabels(names)
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0, 1.1)
axes[1].legend()

# Plot 3: Latency vs accuracy (efficiency frontier)
for i, (n, e, lat) in enumerate(zip(names, exact_accs, latencies)):
    axes[2].scatter(lat, e, color=colors[i], s=150, zorder=5, label=n)
    axes[2].annotate(n, (lat, e), textcoords="offset points", xytext=(6, 4), fontsize=10)
axes[2].set_title("Accuracy–Latency Frontier", fontsize=13, fontweight="bold")
axes[2].set_xlabel("Latency (ms / puzzle)")
axes[2].set_ylabel("Exact Match Accuracy")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("reasoning_decay.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: reasoning_decay.png")

---
## Section 6 — Recursive Depth × Quantization Grid

**Question:** Does recursive depth act as a buffer against quantization noise?
At INT4 precision, does using more recursion steps (`n`) partially restore accuracy?

We sweep `n_cycles ∈ {1, 2, 4, 6, 8}` at inference time across all quantization levels.

In [ ]:
# ── 6.1  Recursive depth sweep at inference time ─────────────────────────────

# @torch.no_grad()
# def evaluate_at_depth(model, loader, trm_ref, n_cycles_override, n_sup=16):
#     """Run evaluation with n_cycles_override inner recursions at inference."""
#     # Temporarily override n_cycles in trm_ref for the init_carry shape;
#     # the model's actual layers are used regardless.
#     model.eval()
#     exact_correct = cell_correct = total = 0

#     for x_batch, y_true in loader:
#         B = x_batch.size(0)
#         y_carry = torch.zeros(B, trm_ref.seq_len, dtype=torch.long)
#         z_carry = trm_ref.z_init.expand(B, -1, -1).clone().cpu()

#         for sup_step in range(n_sup):
#             x_emb = model.x_embed(x_batch)
#             y_emb = model.y_embed(y_carry)

#             # No-grad passes  (T-1)
#             for _ in range(model.T_cycles - 1):
#                 for _ in range(n_cycles_override):
#                     z_carry = model._latent_step(x_emb, y_emb, z_carry)
#                 y_emb = model._answer_step(y_emb, z_carry)

#             # Final pass
#             for _ in range(n_cycles_override):
#                 z_carry = model._latent_step(x_emb, y_emb, z_carry)
#             y_emb = model._answer_step(y_emb, z_carry)

#             logits = model.lm_head(y_emb)
#             y_pred = logits.argmax(-1)
#             q      = model.halt_prob(y_emb)
#             y_carry = y_pred
#             z_carry = z_carry.detach()
#             if q.mean().item() > 0.5 and sup_step > 0:
#                 break

#         exact_correct += (y_pred == y_true).all(dim=-1).sum().item()
#         cell_correct  += (y_pred == y_true).sum().item()
#         total         += B

#     return exact_correct / total, cell_correct / (total * trm_ref.seq_len)
@torch.no_grad()
def evaluate_at_depth(model, loader, trm_ref, n_cycles_override, n_sup=16):
    model.eval()
    exact_correct = cell_correct = total = 0
    for x_batch, y_true in loader:
        B = x_batch.size(0)
        y_carry = torch.zeros(B, trm_ref.seq_len, dtype=torch.long)
        z_carry = trm_ref.z_init.expand(B, -1, -1).clone().cpu()

        x_emb = model.x_embed(x_batch)          # ← MOVED HERE, outside sup_step loop

        for sup_step in range(n_sup):
            y_emb = model.y_embed(y_carry)       # y_carry changes each step, so this stays
            # No-grad passes (T-1)
            for _ in range(model.T_cycles - 1):
                for _ in range(n_cycles_override):
                    z_carry = model._latent_step(x_emb, y_emb, z_carry)
                y_emb = model._answer_step(y_emb, z_carry)
            # Final pass
            for _ in range(n_cycles_override):
                z_carry = model._latent_step(x_emb, y_emb, z_carry)
            y_emb  = model._answer_step(y_emb, z_carry)
            logits = model.lm_head(y_emb)
            y_pred = logits.argmax(-1)
            q      = model.halt_prob(y_emb)
            y_carry = y_pred
            z_carry = z_carry.detach()
            if torch.sigmoid(q).mean().item() > 0.5 and sup_step > 0:
                break
            # if q.mean().item() > 0.5 and sup_step > 0:
            #     break

        exact_correct += (y_pred == y_true).all(dim=-1).sum().item()
        cell_correct  += (y_pred == y_true).sum().item()
        total         += B
    return exact_correct / total, cell_correct / (total * trm_ref.seq_len)

N_CYCLES_SWEEP = [1, 2, 4, 6] if SMOKE_TEST else [1, 2, 3, 4, 6, 8]
grid_results   = {}   # (quant_level, n_cycles) → (exact, cell)

models_to_sweep = [("FP32", trm_fp32), ("INT4 (fake)", trm_int4)]
if not SMOKE_TEST:
    models_to_sweep.insert(1, ("INT8", trm_int8))

print(f"{'Quant':<14} {'n_cycles':<10} {'Exact Acc':>10} {'Cell Acc':>10}")
print("-" * 50)
for qname, qmodel in models_to_sweep:
    for nc in N_CYCLES_SWEEP:
        exact, cell = evaluate_at_depth(qmodel, test_loader, trm_ref=trm,
                                        n_cycles_override=nc)
        grid_results[(qname, nc)] = (exact, cell)
        print(f"{qname:<14} {nc:<10} {exact:>10.4f} {cell:>10.4f}")

In [ ]:
# ── 6.2  Plot the depth × quantization grid ──────────────────────────────────
quant_levels = list(dict.fromkeys(k[0] for k in grid_results))
line_colors  = {"FP32": "#2196F3", "INT8": "#4CAF50", "INT4 (fake)": "#FF5722"}
line_styles  = {"FP32": "-",  "INT8": "--", "INT4 (fake)": ":"}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

for qname in quant_levels:
    nc_vals = [nc for (qn, nc) in grid_results if qn == qname]
    exact_v = [grid_results[(qname, nc)][0] for nc in nc_vals]
    cell_v  = [grid_results[(qname, nc)][1] for nc in nc_vals]
    c, ls   = line_colors[qname], line_styles[qname]
    ax1.plot(nc_vals, exact_v, marker="o", color=c, ls=ls, label=qname, lw=2)
    ax2.plot(nc_vals, cell_v,  marker="s", color=c, ls=ls, label=qname, lw=2)

for ax, title in [(ax1, "Exact Match Accuracy"), (ax2, "Cell-Level Accuracy")]:
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlabel("n_cycles (inner recursion depth)")
    ax.set_ylabel("Accuracy")
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xticks(N_CYCLES_SWEEP)

plt.suptitle("Recursive Depth vs Quantization Level", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("depth_quant_grid.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: depth_quant_grid.png")

---
## Section 7 — Similarity-Score Loss (Novel Contribution)

**Idea (from your proposal):** Force recursive steps to be *dissimilar* to each other by
penalising the cosine similarity between consecutive latent states `z_t` and `z_{t+1}`.
This encourages each cycle to specialise on a different aspect of the reasoning chain.

**Hypothesis A:** Dissimilarity regularisation → faster convergence across supervision steps.  
**Hypothesis B:** More specialised steps → fewer cycles needed to reach the same accuracy  
  → more compression-tolerant model (each step carries less redundancy).

We compare TRM trained **with** and **without** this loss.

In [ ]:
# ── 7.1  Similarity-Score Loss ────────────────────────────────────────────────

def similarity_penalty(z_states: list, reduction: str = "mean") -> torch.Tensor:
    """
    Compute mean cosine similarity between consecutive latent states.
    z_states: list of tensors each (B, L, D)
    Returns a scalar penalty ∈ [-1, 1] (higher = more redundant).
    """
    if len(z_states) < 2:
        return torch.tensor(0.0)

    sims = []
    for z_t, z_tp1 in zip(z_states[:-1], z_states[1:]):
        # Pool over L, normalise over D
        a = z_t.mean(1)    # (B, D)
        b = z_tp1.mean(1)  # (B, D)
        cos = F.cosine_similarity(a, b, dim=-1)  # (B,)
        sims.append(cos)

    sims = torch.stack(sims, dim=0)  # (n_steps-1, B)
    return sims.mean()               # scalar


def train_trm_with_sim_loss(model: TinyRecursiveModel,
                            train_loader: DataLoader,
                            val_loader:   DataLoader,
                            n_epochs:     int   = 200,
                            lr:           float = 1e-4,
                            weight_decay: float = 1.0,
                            n_sup:        int   = 8,
                            sim_lambda:   float = 0.1,  # weight on similarity penalty
                            log_interval: int   = 20):
    """
    Extended training loop that includes the similarity-score regularisation term.
    sim_lambda: weight of the dissimilarity penalty (higher → more specialisation).
    """
    opt    = torch.optim.AdamW(model.parameters(), lr=lr,
                               weight_decay=weight_decay, betas=(0.9, 0.95))
    sched  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)
    ema    = EMA(model, decay=0.999)
    history = {"train_loss": [], "sim_penalty": [], "val_acc": []}

    for epoch in range(1, n_epochs + 1):
        model.train()
        epoch_loss = epoch_sim = 0.0

        for x_batch, y_true in train_loader:
            x_batch = x_batch.to(DEVICE)
            y_true  = y_true.to(DEVICE)
            B       = x_batch.size(0)

            y_carry = torch.zeros(B, model.seq_len, dtype=torch.long, device=DEVICE)
            z_carry = model.init_carry(B).to(DEVICE)

            total_loss = sim_total = torch.tensor(0.0, device=DEVICE)

            for step in range(n_sup):
                x_emb = model.x_embed(x_batch)
                y_emb = model.y_embed(y_carry)

                # ── Collect latent states for similarity penalty ──────────────
                z_trajectory = [z_carry]

                # No-grad passes (T-1)
                with torch.no_grad():
                    for _ in range(model.T_cycles - 1):
                        for _ in range(model.n_cycles):
                            z_carry = model._latent_step(x_emb, y_emb, z_carry)
                            z_trajectory.append(z_carry)
                        y_emb = model._answer_step(y_emb, z_carry)

                # Grad pass
                for _ in range(model.n_cycles):
                    z_carry = model._latent_step(x_emb, y_emb, z_carry)
                    z_trajectory.append(z_carry)
                y_emb = model._answer_step(y_emb, z_carry)

                logits = model.lm_head(y_emb)
                y_pred = logits.argmax(-1)
                q      = model.halt_prob(y_emb)

                # ── Losses ────────────────────────────────────────────────────
                ce_loss  = F.cross_entropy(logits.reshape(-1, model.vocab_size),
                                            y_true.reshape(-1))
                act_loss = F.binary_cross_entropy(q, (y_pred == y_true).all(-1).float())
                # Similarity penalty: maximise dissimilarity (minimise cosine sim)
                sim_pen  = similarity_penalty(z_trajectory)
                total_loss = total_loss + ce_loss + 0.1 * act_loss + sim_lambda * sim_pen
                sim_total  = sim_total + sim_pen

                y_carry = y_pred.detach()
                z_carry = z_carry.detach()
                if q.mean().item() > 0.5 and step > 0:
                    break

            opt.zero_grad()
            total_loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            ema.update()
            epoch_loss += total_loss.item()
            epoch_sim  += sim_total.item()

        sched.step()
        avg_loss = epoch_loss / len(train_loader)
        avg_sim  = epoch_sim  / len(train_loader)
        history["train_loss"].append(avg_loss)
        history["sim_penalty"].append(avg_sim)

        if epoch % log_interval == 0 or epoch == n_epochs:
            ema.apply_shadow()
            exact, _ = evaluate(model, val_loader)
            ema.restore()
            history["val_acc"].append(exact)
            print(f"Epoch {epoch:4d}/{n_epochs} | loss={avg_loss:.4f} "
                  f"| sim_pen={avg_sim:.4f} | val_exact={exact:.3f}")

    return history, ema

In [ ]:
# ── 7.2  Train with similarity loss and compare ───────────────────────────────



print("Training TRM + Similarity-Score Loss...")
trm_sim = TinyRecursiveModel(hidden_size=H).to(DEVICE)
# After creating trm:
# trm = TinyRecursiveModel(hidden_size=H).to(DEVICE)
trm_sim = torch.compile(trm_sim, mode="max-autotune")  # H100: use max-autotune
if NUM_GPUS > 1:
    trm_sim = torch.nn.DataParallel(trm_sim)
trm_sim = trm_sim.to(DEVICE)

# Initialise with the same weights as the base TRM for fair comparison
trm_sim.load_state_dict(copy.deepcopy(trm.state_dict()))

sim_history, sim_ema = train_trm_with_sim_loss(
    trm_sim, train_loader, val_loader,
    n_epochs=N_EPOCHS,
    sim_lambda=0.1,
    log_interval=max(1, N_EPOCHS // 10)
)

sim_ema.apply_shadow()
sim_exact, sim_cell = evaluate(trm_sim.cpu(), test_loader)
sim_ema.restore()

print(f"\nBaseline TRM  | exact={fp32_exact:.4f} | cell={fp32_cell:.4f}")
print(f"TRM + SimLoss | exact={sim_exact:.4f} | cell={sim_cell:.4f}")
print(f"Delta         | exact={sim_exact - fp32_exact:+.4f} | cell={sim_cell - fp32_cell:+.4f}")

In [ ]:
# ── 7.3  Visualise similarity evolution across training ───────────────────────
if sim_history["sim_penalty"]:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    axes[0].plot(sim_history["train_loss"], label="TRM+SimLoss", color="#FF5722")
    if history:
        axes[0].plot(history["train_loss"],     label="Baseline TRM", color="#2196F3", ls="--")
    axes[0].set_title("Train Loss Comparison", fontsize=13, fontweight="bold")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].set_yscale("log")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(sim_history["sim_penalty"], color="#9C27B0", lw=2)
    axes[1].set_title("Mean Cosine Similarity Between Recursive Steps",
                      fontsize=12, fontweight="bold")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Cosine Similarity (↓ = more specialised)")
    axes[1].axhline(0, ls="--", c="gray", alpha=0.5)
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("similarity_loss_analysis.png", dpi=150, bbox_inches="tight")
    plt.show()

# ── 7.4  Test: does SimLoss model tolerate INT4 better? ──────────────────────
print("\nQuantization tolerance comparison (INT4):")
trm_sim_cpu   = trm_sim.cpu()
trm_sim_int4  = quantize_int4_fake(trm_sim_cpu)

base_fp32_e, _ = evaluate_cpu(trm_fp32,     test_loader, trm_ref=trm)
base_int4_e, _ = evaluate_cpu(trm_int4,     test_loader, trm_ref=trm)
sim_fp32_e, _  = evaluate_cpu(trm_sim_cpu,  test_loader, trm_ref=trm)
sim_int4_e, _  = evaluate_cpu(trm_sim_int4, test_loader, trm_ref=trm)

print(f"  Baseline TRM:  FP32={base_fp32_e:.4f} → INT4={base_int4_e:.4f}  "
      f"(drop={base_fp32_e - base_int4_e:+.4f})")
print(f"  TRM+SimLoss:   FP32={sim_fp32_e:.4f} → INT4={sim_int4_e:.4f}  "
      f"(drop={sim_fp32_e - sim_int4_e:+.4f})")

---
## Section 8 — Model Size & SRAM Footprint Estimator

This section maps the compression landscape to realistic embedded hardware targets.

| Target device | SRAM budget | Example MCU |
|---|---|---|
| Ultra-edge | < 256 KB | STM32F4, ESP32 |
| Standard TinyML | < 1 MB  | Arduino Portenta H7, ESP32-S3 |
| Micro-edge | < 4 MB  | Raspberry Pi Zero, NRF9160 |

In [ ]:
# ── 8.1  Comprehensive model footprint analysis ───────────────────────────────

def activation_memory_kb(model: TinyRecursiveModel, batch_size: int = 1,
                          bits: int = 32) -> float:
    """
    Estimate peak activation memory during inference.
    A rough bound: L * D * n_cycles * T_cycles * batch_size * bytes.
    """
    bytes_per_val = bits / 8
    # Two tensors in flight: z_carry, y_emb at any time
    activations = 2 * model.seq_len * model.hidden_size * batch_size * bytes_per_val
    return activations / 1024


def full_inference_memory_kb(model: TinyRecursiveModel, bits: int = 32,
                              batch_size: int = 1) -> dict:
    params_kb  = estimate_model_size_kb(model, bits)
    activ_kb   = activation_memory_kb(model, batch_size, bits)
    total_kb   = params_kb + activ_kb
    return {
        "params_kb":  params_kb,
        "activ_kb":   activ_kb,
        "total_kb":   total_kb,
        "fits_256KB":  total_kb < 256,
        "fits_1MB":    total_kb < 1024,
        "fits_4MB":    total_kb < 4096,
    }


# Sweep across hidden sizes and precisions for a publishable table
hidden_sizes  = [32, 64, 128, 256, 512]
precisions    = [("FP32", 32), ("INT8", 8), ("INT4", 4), ("INT2", 2)]

rows = []
for H_sz in hidden_sizes:
    probe = TinyRecursiveModel(hidden_size=H_sz)  # just for counting
    n_params = count_params(probe)
    for prec_name, bits in precisions:
        mem = full_inference_memory_kb(probe, bits=bits)
        rows.append({
            "hidden_size":  H_sz,
            "n_params_M":   n_params / 1e6,
            "precision":    prec_name,
            "params_KB":    mem["params_kb"],
            "activ_KB":     mem["activ_kb"],
            "total_KB":     mem["total_kb"],
            "total_MB":     mem["total_kb"] / 1024,
            "fits_256KB":   "✓" if mem["fits_256KB"] else "✗",
            "fits_1MB":     "✓" if mem["fits_1MB"]   else "✗",
        })

df = pd.DataFrame(rows)
print(df.to_string(index=False, float_format="{:.3f}".format))

In [ ]:
# ── 8.2  SRAM footprint visualisation ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))

palette = {"FP32": "#2196F3", "INT8": "#4CAF50", "INT4": "#FF9800", "INT2": "#F44336"}
markers = {"FP32": "o", "INT8": "s", "INT4": "^", "INT2": "D"}

for prec_name, _ in precisions:
    sub = df[df["precision"] == prec_name]
    ax.plot(sub["hidden_size"], sub["total_MB"],
            marker=markers[prec_name], color=palette[prec_name],
            label=prec_name, lw=2, markersize=8)

# Hardware budget lines
ax.axhline(0.25, ls=":",  color="red",    alpha=0.7, lw=1.5, label="256 KB (STM32F4)")
ax.axhline(1.0,  ls="--", color="orange", alpha=0.7, lw=1.5, label="1 MB (ESP32-S3)")
ax.axhline(4.0,  ls="-.", color="gray",   alpha=0.7, lw=1.5, label="4 MB (RPi Zero)")

ax.set_title("TRM SRAM Footprint vs Hidden Size & Precision",
             fontsize=14, fontweight="bold")
ax.set_xlabel("Hidden Size (D)", fontsize=12)
ax.set_ylabel("Total Memory (MB)", fontsize=12)
ax.set_yscale("log")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.3f}"))
ax.legend(ncol=2, fontsize=10)
ax.grid(True, which="both", alpha=0.3)

plt.tight_layout()
plt.savefig("sram_footprint.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: sram_footprint.png")

In [ ]:
# ── 8.3  Accuracy vs SRAM: the key publishable plot ──────────────────────────
# Populate this dict with real results from Sections 5 & 6 after a full training run.
# The values below are illustrative placeholders — replace with your actual numbers.

acc_vs_sram = [
    # (label,              approx_total_MB, exact_acc)   ← fill from Sections 5–6
    ("FP32  (H=512)",     26.8,  fp32_exact),
    ("INT8  (H=512)",      6.7,  results.get("INT8", {}).get("exact_acc", 0)),
    ("INT4  (H=512)",      3.4,  results.get("INT4 (fake)", {}).get("exact_acc", 0)),
    ("FP32  (H=128)",      1.7,  None),   # run evaluate() after training H=128 model
    ("INT4  (H=128)",      0.2,  None),   # sub-256 KB target
]

fig, ax = plt.subplots(figsize=(10, 6))
for label, mem_mb, acc in acc_vs_sram:
    if acc is None:
        ax.scatter(mem_mb, 0, marker="x", color="gray", s=100)
        ax.annotate(f"{label}\n(not run)", (mem_mb, 0.01),
                    fontsize=8, ha="center", color="gray")
    else:
        ax.scatter(mem_mb, acc, s=150, zorder=5)
        ax.annotate(f"{label}\n{acc:.3f}", (mem_mb, acc),
                    textcoords="offset points", xytext=(8, 4), fontsize=9)

ax.axvline(1.0, ls="--", color="orange", alpha=0.7, lw=1.5, label="1 MB SRAM target")
ax.axvline(0.25, ls=":", color="red",    alpha=0.7, lw=1.5, label="256 KB SRAM target")
ax.set_title("Accuracy vs SRAM Footprint — Edge-AGI Feasibility",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Model Memory (MB, log scale)")
ax.set_ylabel("Exact Match Accuracy")
ax.set_xscale("log")
ax.set_xlim(0.05, 100)
ax.set_ylim(-0.05, 1.05)
ax.legend()
ax.grid(True, which="both", alpha=0.3)

plt.tight_layout()
plt.savefig("accuracy_vs_sram.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: accuracy_vs_sram.png")

---
## Summary & Next Steps

### Results collected
| Experiment | Key metric | Cell to fill |
|---|---|---|
| Reasoning Decay (§5) | Δexact FP32→INT4 | `results` dict |
| Depth × Quant (§6) | Best n_cycles per precision | `grid_results` dict |
| Similarity Loss (§7) | Δexact vs baseline | Printed comparison |
| SRAM feasibility (§8) | Fits <1 MB? | `df` DataFrame |

### Recommended next steps for a full paper

1. **Full training** (§3 command): 50k epochs on Sudoku-Extreme-1K with 1000 augmentations → ~87% FP32 checkpoint.
2. **Quantisation-Aware Training (QAT)**: Rather than post-training quantization, train with INT4 fake quant from the start — almost certainly recovers much of the accuracy gap.
3. **Structured pruning sweep**: Remove entire attention heads / MLP blocks; compare with n_cycles reduction.
4. **On-device validation**: Export to TFLite Micro / ONNX and measure actual latency on an ESP32-S3 or Cortex-M7.
5. **Similarity loss ablation table**: Vary `sim_lambda ∈ {0.0, 0.01, 0.05, 0.1, 0.5}` and report accuracy + INT4 decay.
6. **Extend to ARC-AGI** (requires 4×H100, ~3 days): verify that the decay pattern transfers to harder abstract reasoning.

### Potential paper title
*"Quantized Recursion: Evaluating Abstract Reasoning Under Extreme Compression in Tiny Recursive Models"*

---
*Generated with reference to: Jolicoeur-Martineau (2025), "Less is More: Recursive Reasoning with Tiny Networks", arXiv:2510.04871*